# Olist 데이터 프로파일링

9개 원본 테이블의 구조, 품질, 관계를 체계적으로 검토한다.

**분석 목표:**
- 데이터셋 규모 및 결측치 현황 파악
- 테이블 간 참조 무결성(Referential Integrity) 검증
- 핵심 변수의 분포 및 이상치 확인
- 시간/결제/지역별 기초 패턴 탐색

In [1]:
import pandas as pd
import numpy as np
import os

data_path = '../data/'
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
print(f"총 {len(csv_files)}개 파일")
for f in csv_files:
    print(f)


총 14개 파일
order_reviews.csv
ML_olist.csv
olist_sellers_dataset.csv
product_category_name_translation.csv
merged_olist.csv
risk_report_result.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_customers_dataset.csv
seller_damage_cost.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv


## 1. 데이터셋 요약 통계

In [2]:
datasets_info = []
for file in csv_files:
    df = pd.read_csv(os.path.join(data_path, file))
    datasets_info.append({
        '파일명': file,
        '행수': len(df),
        '열수': len(df.columns),
        '결측치': df.isnull().sum().sum()
    })
    
info_df = pd.DataFrame(datasets_info)
info_df


,파일명,행수,열수,결측치
0,order_reviews.csv,99224,3,56518
1,ML_olist.csv,64850,14,36990
2,olist_sellers_dataset.csv,3095,4,0
3,product_category_name_translation.csv,71,2,0
4,merged_olist.csv,64850,46,43493
5,risk_report_result.csv,1420,3,0
6,olist_orders_dataset.csv,99441,8,4908
7,olist_order_items_dataset.csv,112650,7,0
8,olist_customers_dataset.csv,99441,5,0
9,seller_damage_cost.csv,239,5,0


In [3]:
for file in csv_files:
    print()
    print('=' * 80)
    df = pd.read_csv(os.path.join(data_path, file))
    print(f"\n{file}")
    display(df.head(3))
    print(f"dtypes:\n{df.dtypes}")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"\n결측치:\n{missing[missing > 0]}")




order_reviews.csv


,order_id,review_score,review_comment_message
0,73fc7af87114b39712e6da79b0a377eb,4,NaN
1,a548910a1c6147796b98fdf73dbeba33,5,NaN
2,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN


dtypes:
order_id                    str
review_score              int64
review_comment_message      str
dtype: object

결측치:
review_comment_message    56518
dtype: int64




ML_olist.csv


,order_id,order_approved_at,order_delivered_carrier_date,seller_id,shipping_limit_date,product_category_name_english,review_score,review_comment_message,has_text_review,seller_processing_days,is_logistics_fault,seller_delay_days,processing_days_diff,order_purchase_timestamp
0,53cdb2fc8bc7dce0b6741e2150273451,2018-07-26 03:24:27,2018-07-26 14:31:00,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,perfumery,4.0,Muito bom o produto.,True,0.46,False,-3.54,-1.85,2018-07-24 20:41:37
1,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:55:23,2018-08-08 13:50:00,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,auto,5.0,NaN,False,0.20,False,-4.80,-2.09,2018-08-08 08:38:49
2,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 22:10:13,2017-07-11 14:58:04,8581055ce74af1daba164fdbd55a40de,2017-07-13 22:10:13,auto,4.0,NaN,False,1.70,False,-2.30,-0.59,2017-07-09 21:57:05


...

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


dtypes:
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object



결측치:
review_comment_title      87656
review_comment_message    58247
dtype: int64


olist_products_dataset.csv


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


dtypes:
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

결측치:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


## 2. 기본 통계 및 분포 확인

In [4]:
orders_df = pd.read_csv('../data/olist_orders_dataset.csv')
items_df = pd.read_csv('../data/olist_order_items_dataset.csv')
customers_df = pd.read_csv('../data/olist_customers_dataset.csv')
payments_df = pd.read_csv('../data/olist_order_payments_dataset.csv')
reviews_df = pd.read_csv('../data/olist_order_reviews_dataset.csv')
products_df = pd.read_csv('../data/olist_products_dataset.csv')
sellers_df = pd.read_csv('../data/olist_sellers_dataset.csv')

# 각 테이블별 기본 통계 출력 (행 수, 주요 컬럼 분포)
print("orders:", len(orders_df), orders_df['order_status'].value_counts())
print('=' * 80)
print("items:", len(items_df), f"price mean: {items_df['price'].mean():.2f}")
print('=' * 80)
print("customers:", len(customers_df), customers_df['customer_state'].value_counts().head())
print('=' * 80)
print("payments:", len(payments_df), payments_df['payment_type'].value_counts())
print('=' * 80)
print("reviews:", len(reviews_df), reviews_df['review_score'].value_counts().sort_index())
print('=' * 80)
print("products:", len(products_df))
print('=' * 80)
print("sellers:", len(sellers_df))

orders: 99441 order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
items: 112650 price mean: 120.65
customers: 99441 customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: count, dtype: int64
payments: 103886 payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
reviews: 99224 review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64
products: 32951
sellers: 3095


## 2-1. 테이블 간 참조 무결성 검증

9개 테이블의 키 매칭 관계(1:1, 1:N, N:M)를 검증한다.

In [5]:
# 테이블 간 키 매칭 확인 (관계 무결성 검증)
def check_integrity(left_df, left_key, right_df, right_key, relation_name):
    """테이블 간 무결성 검증 함수"""
    left_set = set(left_df[left_key])
    right_set = set(right_df[right_key])
    
    # 기본 통계
    left_unique = len(left_set)
    right_unique = len(right_set)
    left_total = len(left_df)
    right_total = len(right_df)
    intersection = len(left_set & right_set)
    left_only = len(left_set - right_set)
    right_only = len(right_set - left_set)
    
    # 중복 확인
    left_duplicates = left_df[left_key].duplicated().sum()
    right_duplicates = right_df[right_key].duplicated().sum()
    
    print(f"\n{'='*80}")
    print(f"관계: {relation_name}")
    print(f"  왼쪽 테이블: 총 {left_total:,}행, 고유 키 {left_unique:,}개")
    print(f"  오른쪽 테이블: 총 {right_total:,}행, 고유 키 {right_unique:,}개")
    print(f"  교집합: {intersection:,}")
    print(f"  왼쪽에만 존재: {left_only:,}")
    print(f"  오른쪽에만 존재: {right_only:,}")
    print(f"  왼쪽 중복 키: {left_duplicates:,}")
    print(f"  오른쪽 중복 키: {right_duplicates:,}")
    print()
    
    # 관계 유형 및 해석
    if left_duplicates == 0 and right_duplicates == 0:
        print(f"1:1 관계 (각 키가 양쪽 테이블에서 모두 고유)")
    elif left_duplicates == 0 and right_duplicates > 0:
        print(f"1:N 관계 (왼쪽 1개 키 -> 오른쪽 여러 행)")
        print(f"오른쪽 중복 키 {right_duplicates:,}개")
    elif left_duplicates > 0 and right_duplicates == 0:
        print(f"N:1 관계 (왼쪽 여러 행 -> 오른쪽 1개 키)")
        print(f"왼쪽 중복 키 {left_duplicates:,}개")
    else:
        print(f"N:M 관계 (양쪽 모두 중복 허용)")
    
    if left_only > 0:
        print(f"왼쪽에만 존재하는 {left_only:,}개")
    if right_only > 0:
        print(f"오른쪽에만 존재하는 {right_only:,}개")

# 각 관계별 무결성 검증
check_integrity(orders_df, 'order_id', items_df, 'order_id', 'orders - order_items')
check_integrity(orders_df, 'customer_id', customers_df, 'customer_id', 'orders - customers')
check_integrity(orders_df, 'order_id', payments_df, 'order_id', 'orders - payments')
check_integrity(orders_df, 'order_id', reviews_df, 'order_id', 'orders - reviews')
check_integrity(items_df, 'product_id', products_df, 'product_id', 'order_items - products')
check_integrity(items_df, 'seller_id', sellers_df, 'seller_id', 'order_items - sellers')
print(f"\n{'='*80}")



관계: orders - order_items
  왼쪽 테이블: 총 99,441행, 고유 키 99,441개
  오른쪽 테이블: 총 112,650행, 고유 키 98,666개
  교집합: 98,666
  왼쪽에만 존재: 775
  오른쪽에만 존재: 0
  왼쪽 중복 키: 0
  오른쪽 중복 키: 13,984

1:N 관계 (왼쪽 1개 키 -> 오른쪽 여러 행)
오른쪽 중복 키 13,984개
왼쪽에만 존재하는 775개

관계: orders - customers
  왼쪽 테이블: 총 99,441행, 고유 키 99,441개
  오른쪽 테이블: 총 99,441행, 고유 키 99,441개
  교집합: 99,441
  왼쪽에만 존재: 0
  오른쪽에만 존재: 0
  왼쪽 중복 키: 0
  오른쪽 중복 키: 0

1:1 관계 (각 키가 양쪽 테이블에서 모두 고유)



관계: orders - payments
  왼쪽 테이블: 총 99,441행, 고유 키 99,441개
  오른쪽 테이블: 총 103,886행, 고유 키 99,440개
  교집합: 99,440
  왼쪽에만 존재: 1
  오른쪽에만 존재: 0
  왼쪽 중복 키: 0
  오른쪽 중복 키: 4,446

1:N 관계 (왼쪽 1개 키 -> 오른쪽 여러 행)
오른쪽 중복 키 4,446개
왼쪽에만 존재하는 1개



관계: orders - reviews
  왼쪽 테이블: 총 99,441행, 고유 키 99,441개
  오른쪽 테이블: 총 99,224행, 고유 키 98,673개
  교집합: 98,673
  왼쪽에만 존재: 768
  오른쪽에만 존재: 0
  왼쪽 중복 키: 0
  오른쪽 중복 키: 551

1:N 관계 (왼쪽 1개 키 -> 오른쪽 여러 행)
오른쪽 중복 키 551개
왼쪽에만 존재하는 768개

관계: order_items - products
  왼쪽 테이블: 총 112,650행, 고유 키 32,951개
  오른쪽 테이블: 총 32,951행, 고유 키 32,951개
  교집합: 32,951
  왼쪽에만 존재: 0
  오른쪽에만 존재: 0
  왼쪽 중복 키: 79,699
  오른쪽 중복 키: 0

N:1 관계 (왼쪽 여러 행 -> 오른쪽 1개 키)
왼쪽 중복 키 79,699개

관계: order_items - sellers
  왼쪽 테이블: 총 112,650행, 고유 키 3,095개
  오른쪽 테이블: 총 3,095행, 고유 키 3,095개
  교집합: 3,095
  왼쪽에만 존재: 0
  오른쪽에만 존재: 0
  왼쪽 중복 키: 109,555
  오른쪽 중복 키: 0

N:1 관계 (왼쪽 여러 행 -> 오른쪽 1개 키)
왼쪽 중복 키 109,555개



## 3. 데이터 품질 검사

#### 참고

- set은 수학의 '집합'과 같음
- 가장 큰 특징은 중복을 허용하지 않음
  - 예를 들어 orders_df['order_id']를 하면 수많은 주문 ID가 나오는데, 이를 set()으로 감싸면 중복된 ID는 모두 사라지고 고유한 ID만 남게 됩니다.
- set 자료형끼리 & 기호를 사용하면 교집합을 구한다.
**즉, 양쪽 모두에 존재하는 값만 남긴다.**

- .unique()는 '배열(Array)'을 돌려주고, set()은 '집합(Set)'을 돌려준다
- 교집합(&)이나 차집합(-) 같은 수학 연산은 '집합'에서만 바로 쓸 수 있습니다. 만약 배열에서 차집합을 한다면 숫자뺄셈을 시도한다

In [6]:
orders_without_items = set(orders_df['order_id']) - set(items_df['order_id'])
print(f"주문 항목 없는 주문: {len(orders_without_items)}개")
if orders_without_items:
    print(orders_df[orders_df['order_id'].isin(orders_without_items)]['order_status'].value_counts())
print('=' * 80)

print(f"상품 정보 없는 product_id: {len(set(items_df['product_id']) - set(products_df['product_id']))}개")
print(f"가격 음수: {(items_df['price'] <= 0).sum()}개")
print(f"배송비 음수: {(items_df['freight_value'] < 0).sum()}개")
print('=' * 80)

print(f"결제 금액 0 이하: {(payments_df['payment_value'] <= 0).sum()}개")

pay_counts = payments_df.groupby('order_id').size()
multi_pay_count = (pay_counts > 1).sum()
print(f"여러 번 결제된 주문 수: {multi_pay_count}개")
print('=' * 80)

print(f"리뷰 중복: {reviews_df['order_id'].duplicated().sum()}개")
print(f"리뷰 점수 결측: {reviews_df['review_score'].isnull().sum()}개")
print('=' * 80)

print(f"카테고리 결측: {products_df['product_category_name'].isnull().sum()}개")
print(f"무게 결측: {products_df['product_weight_g'].isnull().sum()}개")


주문 항목 없는 주문: 775개
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64
상품 정보 없는 product_id: 0개


가격 음수: 0개
배송비 음수: 0개
결제 금액 0 이하: 9개
여러 번 결제된 주문 수: 2961개
리뷰 중복: 551개
리뷰 점수 결측: 0개


카테고리 결측: 610개
무게 결측: 2개


In [7]:
print("가격 통계:")
print(items_df['price'].describe())

print(f"\n배송비: mean={items_df['freight_value'].mean():.2f}, median={items_df['freight_value'].median():.2f}")
print(f"결제금액: mean={payments_df['payment_value'].mean():.2f}, median={payments_df['payment_value'].median():.2f}")
print(f"할부횟수: mean={payments_df['payment_installments'].mean():.1f}, max={payments_df['payment_installments'].max()}")
print(f"리뷰점수: mean={reviews_df['review_score'].mean():.2f}")
print(reviews_df['review_score'].value_counts().sort_index())


가격 통계:
count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

배송비: mean=19.99, median=16.26
결제금액: mean=154.10, median=100.00
할부횟수: mean=2.9, max=24
리뷰점수: mean=4.09
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


## 4. 시간별 주문 추이 분석

In [8]:
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'], errors='coerce')
orders_df['order_approved_at'] = pd.to_datetime(orders_df['order_approved_at'], errors='coerce')
orders_df['order_delivered_customer_date'] = pd.to_datetime(orders_df['order_delivered_customer_date'], errors='coerce')
orders_df['order_estimated_delivery_date'] = pd.to_datetime(orders_df['order_estimated_delivery_date'], errors='coerce')

print(f"주문 기간: {orders_df['order_purchase_timestamp'].min()} ~ {orders_df['order_purchase_timestamp'].max()}")

orders_df['order_year_month'] = orders_df['order_purchase_timestamp'].dt.to_period('M')
print("\n월별 주문 수:")
print(orders_df.groupby('order_year_month').size().sort_index())

delivered = orders_df[orders_df['order_status'] == 'delivered'].dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date'])
if len(delivered) > 0:
    delivered['delivery_time'] = (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']).dt.days
    delivered['delivery_delay'] = (delivered['order_delivered_customer_date'] - delivered['order_estimated_delivery_date']).dt.days
    print(f"\n배송 시간: mean={delivered['delivery_time'].mean():.1f}일, median={delivered['delivery_time'].median():.0f}일")
    print(f"배송 지연: mean={delivered['delivery_delay'].mean():.1f}일, 지연률={(delivered['delivery_delay'] > 0).sum() / len(delivered) * 100:.1f}%")


주문 기간: 2016-09-04 21:15:19 ~ 2018-10-17 17:30:18

월별 주문 수:
order_year_month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64

배송 시간: mean=12.1일, median=10일
배송 지연: mean=-11.9일, 지연률=6.8%


## 5. 결제 수단 분석

In [9]:
payment_stats = payments_df.groupby('payment_type').agg({
    'payment_value': ['count', 'mean', 'median'],
    'payment_installments': 'mean'
}).round(2)
print("결제 수단별 통계:")
print(payment_stats)

installment = payments_df[payments_df['payment_installments'] > 1]
print(f"\n할부: {len(installment)}건 ({len(installment)/len(payments_df)*100:.1f}%)")
print(f"할부 평균 금액: {installment['payment_value'].mean():.2f}")
print(f"일시불 평균 금액: {payments_df[payments_df['payment_installments'] == 1]['payment_value'].mean():.2f}")

credit_card = payments_df[payments_df['payment_type'] == 'credit_card']
boleto = payments_df[payments_df['payment_type'] == 'boleto']
print(f"\n신용카드: {len(credit_card)}건 ({len(credit_card)/len(payments_df)*100:.1f}%), 할부율={(credit_card['payment_installments'] > 1).sum() / len(credit_card) * 100:.1f}%")
print(f"Boleto: {len(boleto)}건 ({len(boleto)/len(payments_df)*100:.1f}%), 할부율={(boleto['payment_installments'] > 1).sum() / len(boleto) * 100:.1f}%")


결제 수단별 통계:
             payment_value                 payment_installments
                     count    mean  median                 mean
payment_type                                                   
boleto               19784  145.03   93.89                 1.00
credit_card          76795  163.32  106.87                 3.51
debit_card            1529  142.57   89.30                 1.00
not_defined              3    0.00    0.00                 1.00
voucher               5775   65.70   39.28                 1.00

할부: 51338건 (49.4%)
할부 평균 금액: 196.76
일시불 평균 금액: 112.42



신용카드: 76795건 (73.9%), 할부율=66.9%
Boleto: 19784건 (19.0%), 할부율=0.0%


## 6. 지역별 분포

In [10]:
print("고객 지역 상위 10개:")
print(customers_df['customer_state'].value_counts().head(10))

print("\n판매자 지역 상위 10개:")
print(sellers_df['seller_state'].value_counts().head(10))

orders_customers = orders_df.merge(customers_df[['customer_id', 'customer_state']], on='customer_id', how='left')
state_orders = orders_customers.groupby('customer_state').agg({
    'order_id': 'count',
    'customer_id': 'nunique'
}).rename(columns={'order_id': '주문수', 'customer_id': '고객수'})
state_orders['주문당고객수'] = state_orders['주문수'] / state_orders['고객수']
print("\n지역별 주문:")
print(state_orders.sort_values('주문수', ascending=False).head(10))


고객 지역 상위 10개:
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

판매자 지역 상위 10개:
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64



지역별 주문:
                  주문수    고객수  주문당고객수
customer_state                      
SP              41746  41746     1.0
RJ              12852  12852     1.0
MG              11635  11635     1.0
RS               5466   5466     1.0
PR               5045   5045     1.0
SC               3637   3637     1.0
BA               3380   3380     1.0
DF               2140   2140     1.0
ES               2033   2033     1.0
GO               2020   2020     1.0
